In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import torch

USE_CUDA = torch.cuda.is_available()
DEVICE = "cuda" if USE_CUDA else "cpu"
PIPELINE_DEVICE = 0 if USE_CUDA else -1   # what HF `pipeline(device=...)` expects
TORCH_DTYPE = torch.float16 if USE_CUDA else torch.float32

if USE_CUDA:
    torch.backends.cudnn.benchmark = True  # speeds up fixed-size batch workloads on P100
    print("GPU detected:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — running on CPU. Everything will still work, just slower.")
    print("On Kaggle: Notebook Settings -> Accelerator -> GPU P100 to speed this up.")

print("DEVICE =", DEVICE)


GPU detected: Tesla T4
DEVICE = cuda


In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

In [4]:
!nvidia-smi

Wed Jul  8 15:22:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# EDA

In [5]:
import re
import string

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)


In [6]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [7]:
print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [8]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [9]:
print(train.info())
print(train.isnull().sum())
print(train['answer'].value_counts())
print(train.duplicated().sum())
print(train.head(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB
None
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
0
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   

                                               

# Text Cleaning

In [10]:
from sklearn.model_selection import train_test_split

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

text_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']

for col in text_cols:
    train[col] = train[col].apply(clean_text)
    test[col] = test[col].apply(clean_text)

train[['prompt', 'A']].head()


,prompt,A
0,pick the best possible answer what is martin h...,martin heidegger believes that humans exist wi...
1,what is acceleratorbased lightion fusion,acceleratorbased lightion fusion is a techniqu...
2,determine the correct option what is the term ...,blueshifting
3,select the most accurate option what is martin...,martin heidegger believes that humans exist wi...
4,identify the correct statement what is the con...,simultaneity is relative meaning that two even...


# Train / Validation Split

In [11]:
import re

PROMPT_PREFIXES = [
    r"^pick the best possible answer\s*:?\s*",
    r"^select the most accurate option\s*:?\s*",
    r"^determine the correct option\s*:?\s*",
    r"^identify the correct statement\s*:?\s*",
    r"^choose the correct answer\s*:?\s*",
]

PROMPT_SUFFIXES = [
    r"\s*among the listed options\.?$",
    r"\s*from the following choices\.?$",
    r"\s*based on the given context\.?$",
    r"\s*carefully\.?$",
]

def extract_query(prompt):
    """Strip quiz boilerplate off a prompt, leaving a clean search query."""
    text = str(prompt).strip()
    for pat in PROMPT_PREFIXES:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)
    for pat in PROMPT_SUFFIXES:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)
    return text.strip()

In [12]:
options = ['A', 'B', 'C', 'D', 'E']
tr, val = train_test_split(train, test_size=0.2, random_state=SEED, stratify=train['answer'])
print("Train:", tr.shape, "Validation:", val.shape)

Train: (1600, 8) Validation: (400, 8)


In [13]:
print(train['answer'].value_counts(normalize=True))
print(tr['answer'].value_counts(normalize=True))
print(val['answer'].value_counts(normalize=True))

answer
B    0.2450
C    0.2295
A    0.1845
D    0.1790
E    0.1620
Name: proportion, dtype: float64
answer
B    0.245000
C    0.229375
A    0.184375
D    0.179375
E    0.161875
Name: proportion, dtype: float64
answer
B    0.2450
C    0.2300
A    0.1850
D    0.1775
E    0.1625
Name: proportion, dtype: float64


# mAP@3 Scoring

In [14]:
def apk(actual, predicted, k=3):

    predicted = predicted[:k]
    for i, p in enumerate(predicted):
        if p == actual:
            return 1.0 / (i + 1)
    return 0.0

In [15]:
def mapk(actuals, predictions, k=3):
    """Mean average precision @ k. `predictions` is a list/Series of
    space-separated letter strings, e.g. 'A B C'."""
    scores = [
        apk(actual, pred.split(), k)
        for actual, pred in zip(actuals, predictions)
    ]
    return np.mean(scores)

In [16]:
assert apk('A', ['A', 'B', 'C']) == 1.0
assert apk('B', ['A', 'B', 'C']) == 0.5
assert apk('D', ['A', 'B', 'C']) == 0.0
print("mAP@3 helper functions OK")

mAP@3 helper functions OK


## Weights & Biases setup

In [17]:
import wandb

WANDB_PROJECT = "YourRollNo-t22026" 

def log_run(model_name, score, k=3, extra_config=None):
    """Log a single model's mAP@k validation score as a W&B run."""
    config = {"model": model_name, "k": k}
    if extra_config:
        config.update(extra_config)
    run = wandb.init(project=WANDB_PROJECT, name=model_name, config=config, reinit=True)
    wandb.log({f"mAP@{k}": score})
    run.finish()
    print(f"Logged to W&B: {model_name} -> mAP@{k} = {round(score, 4)}")

## Random baseline

Worth establishing *before* looking at any model: with 5 options and a top-3
guess, a uniformly random ranking already scores fairly high on mAP@3 purely by
chance. Any real method needs to clear this bar to be worth using.

In [18]:
rng = np.random.default_rng(SEED)

random_preds = [
    " ".join(rng.permutation(options)[:3])
    for _ in range(len(val))
]

random_score = mapk(val['answer'], random_preds)
print("Random baseline mAP@3 (validation):", round(random_score, 4))


Random baseline mAP@3 (validation): 0.3208


# TF-IDF + Cosine Similarity

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
all_text = pd.concat([train[c] for c in text_cols] + [test[c] for c in text_cols])

tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf.fit(all_text)

print("Vocabulary size:", len(tfidf.vocabulary_))

Vocabulary size: 2865


In [21]:
def predict_top3_tfidf(df, vectorizer):
  
    prompt_mat = vectorizer.transform(df['prompt'])
    option_mats = {opt: vectorizer.transform(df[opt]) for opt in options}

 
    sims = np.column_stack([
        cosine_similarity(prompt_mat, option_mats[opt]).diagonal()
        for opt in options
    ])  

    rankings = np.argsort(-sims, axis=1)
    preds = [" ".join(np.array(options)[r][:3]) for r in rankings]
    return preds

In [22]:
val_preds_tfidf = predict_top3_tfidf(val, tfidf)
tfidf_score = mapk(val['answer'], val_preds_tfidf)
print("TF-IDF mAP@3 (validation):", round(tfidf_score, 4))
print("Random baseline   mAP@3 (validation):", round(random_score, 4))

TF-IDF mAP@3 (validation): 0.2821
Random baseline   mAP@3 (validation): 0.3208


In [23]:
for i in range(5):
    print("Prompt:", val.iloc[i]['prompt'])
    print("Prediction:", val_preds_tfidf[i])
    print("Actual:", val.iloc[i]['answer'])
    print()

Prompt: what is a phageome
Prediction: A B C
Actual: C

Prompt: identify the correct statement what is the role of cycloidea genes in the evolution of bilateral symmetry carefully
Prediction: A E D
Actual: D

Prompt: determine the correct option what did fresnel predict and verify with regards to total internal reflections
Prediction: D B E
Actual: E

Prompt: pick the best possible answer what is the main focus of the environmental science center at qatar university from the following choices
Prediction: B E D
Actual: D

Prompt: what is the reason that newtons second law cannot be used to calculate the development of a physical system in quantum mechanics
Prediction: A B C
Actual: D



# Word2Vec Embeddings

In [24]:
from gensim.models import Word2Vec

import gensim
print(gensim.__version__)

4.4.0


## Tokenize Text

In [25]:
def tokenize(text):
    return str(text).split()

In [26]:
sentences = []
for col in text_cols:
    sentences.extend(train[col].apply(tokenize).tolist())
    sentences.extend(test[col].apply(tokenize).tolist())

print("Number of sentences:", len(sentences))

Number of sentences: 15000


## Training Word2Vec

In [27]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=SEED
)

print("Vocab size:", len(w2v_model.wv))

Vocab size: 3096


In [28]:
print(w2v_model.wv.vector_size)

100


## Sentence Embeddings

In [29]:
def sentence_vector(text, model):
    words = str(text).split()
    vectors = [model.wv[w] for w in words if w in model.wv]
    if not vectors:
        return np.zeros(model.wv.vector_size)
    return np.mean(vectors, axis=0)

## Cosine Similarity using Word2Vec

In [30]:
from sklearn.metrics.pairwise import cosine_similarity

options = ['A','B','C','D','E']

def predict_top3_w2v(df, model):
    prompt_vecs = np.array([sentence_vector(p, model) for p in df['prompt']])
    option_vecs = {
        opt: np.array([sentence_vector(t, model) for t in df[opt]])
        for opt in options
    }

    sims = np.column_stack([
        cosine_similarity(prompt_vecs, option_vecs[opt]).diagonal()
        for opt in options
    ])

    rankings = np.argsort(-sims, axis=1)
    preds = [" ".join(np.array(options)[r][:3]) for r in rankings]
    return preds

## Evaluate

In [31]:
val_preds_w2v = predict_top3_w2v(val, w2v_model)
w2v_score = mapk(val['answer'], val_preds_w2v)
print("Word2Vec mAP@3 (validation):", round(w2v_score, 4))

Word2Vec mAP@3 (validation): 0.3375


# BERT, RoBERTa and Attention Mechanism

## BERT

BERT (Bidirectional Encoder Representations from Transformers) is a pretrained transformer model that learns contextual representations by processing text bidirectionally. Unlike traditional word embeddings, BERT considers both left and right context when representing each word.

## RoBERTa

RoBERTa (Robustly Optimized BERT Approach) is an improved version of BERT. It is trained on more data, for more iterations, and removes the Next Sentence Prediction objective, resulting in better performance on many NLP tasks.

## Self-Attention

The transformer architecture uses self-attention to determine how much each word should focus on every other word in a sentence. This allows the model to capture long-range dependencies and understand context effectively.

## Context-Aware Embeddings

Unlike Word2Vec, which assigns a fixed vector to every word, transformer models generate contextual embeddings. The same word can have different vector representations depending on its surrounding words, leading to better semantic understanding.

# Transformers

In [32]:
from datasets import Dataset

train_ds = Dataset.from_pandas(tr.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test.reset_index(drop=True))

print(train_ds)
print(train_ds.features)
print(train_ds[0])

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 1600
})
{'id': Value('int64'), 'prompt': Value('string'), 'A': Value('string'), 'B': Value('string'), 'C': Value('string'), 'D': Value('string'), 'E': Value('string'), 'answer': Value('string')}
{'id': 387, 'prompt': 'select the most accurate option what is modified newtonian dynamics mond from the following choices', 'A': 'mond is a principle that explains the behavior of light in the presence of strong gravitational fields it is an alternative to the hypothesis of dark matter in terms of explaining why galaxies do not appear to obey the currently understood laws of physics', 'B': 'mond is a hypothesis that proposes a modification of einsteins framework of general relativity to account for observed properties of galaxies it is an alternative to the hypothesis of dark matter in terms of explaining why galaxies do not appear to obey the currently understood laws of physics', 'C': 'mond is a hypoth

In [33]:
!pip install -q sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

In [34]:
from sentence_transformers import SentenceTransformer

minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [35]:
from transformers import AutoTokenizer

minilm_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

# quick look at how the raw (uncleaned) prompts tokenize — no lowercasing/punctuation
# stripping needed here, MiniLM was trained on normal text
sample_ids = minilm_tokenizer(val['prompt'].iloc[0], truncation=True, max_length=64)["input_ids"]
print("Example token ids:", sample_ids[:12])
print("Decoded back:", minilm_tokenizer.decode(sample_ids))

prompt_token_lengths = [
    len(minilm_tokenizer(p, truncation=True, max_length=64)["input_ids"])
    for p in val['prompt']
]
print("Prompt token length — min/mean/max:",
      min(prompt_token_lengths),
      round(sum(prompt_token_lengths) / len(prompt_token_lengths), 1),
      max(prompt_token_lengths))

Example token ids: [101, 2054, 2003, 1037, 6887, 4270, 8462, 102]
Decoded back: [CLS] what is a phageome [SEP]
Prompt token length — min/mean/max: 6 22.6 62


In [36]:
def predict_top3_transformer(df, model, batch_size=64):
    prompt_emb = model.encode(
        df['prompt'].tolist(),
        convert_to_numpy=True,
        batch_size=batch_size,
        show_progress_bar=False,
    )
    option_embs = {
        opt: model.encode(
            df[opt].tolist(),
            convert_to_numpy=True,
            batch_size=batch_size,
            show_progress_bar=False,
        )
        for opt in options
    }

    sims = np.column_stack([
        cosine_similarity(prompt_emb, option_embs[opt]).diagonal()
        for opt in options
    ])

    rankings = np.argsort(-sims, axis=1)
    preds = [" ".join(np.array(options)[r][:3]) for r in rankings]
    return preds

In [37]:
# note: using `val` here, NOT `val_clean` — MiniLM should see the original text
val_preds_transformer = predict_top3_transformer(val, minilm)
transformer_score = mapk(val['answer'], val_preds_transformer)
print("MiniLM mAP@3 (validation):", round(transformer_score, 4))

MiniLM mAP@3 (validation): 0.3925


## Comparing the embedding-based models so far

In [38]:
results = pd.DataFrame({
    "Model": ["Random baseline", "TF-IDF", "Word2Vec", "MiniLM"],
    "mAP@3 (validation)": [random_score, tfidf_score, w2v_score, transformer_score],
}).sort_values("mAP@3 (validation)", ascending=False).reset_index(drop=True)

results

,Model,mAP@3 (validation)
0,MiniLM,0.392500
1,Word2Vec,0.337500
2,Random baseline,0.320833
3,TF-IDF,0.282083


# Zero-Shot Classification

In [39]:
from transformers import AutoTokenizer as NLITokenizer, AutoModelForSequenceClassification

NLI_MODEL_NAME = "facebook/bart-large-mnli"

nli_tokenizer = NLITokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME,
    torch_dtype=TORCH_DTYPE,
).to(DEVICE)
nli_model.eval()

# bart-large-mnli label order: 0=contradiction, 1=neutral, 2=entailment
ENTAILMENT_ID = nli_model.config.label2id.get("entailment", 2)
print("Entailment label id:", ENTAILMENT_ID, "| device:", DEVICE, "| dtype:", TORCH_DTYPE)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Entailment label id: 2 | device: cuda | dtype: torch.float16


In [40]:
@torch.no_grad()
def batched_entailment_scores(premises, hypotheses, batch_size=64, max_length=256):
    """Return the raw entailment logit for each (premise, hypothesis) pair."""
    scores = []
    for i in range(0, len(premises), batch_size):
        batch_p = premises[i:i + batch_size]
        batch_h = hypotheses[i:i + batch_size]
        enc = nli_tokenizer(
            batch_p, batch_h,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True,
        ).to(DEVICE)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_CUDA):
            logits = nli_model(**enc).logits

        entail_logits = logits[:, ENTAILMENT_ID].float()
        scores.extend(entail_logits.detach().cpu().tolist())
    return scores

In [41]:
def predict_top3_nli(df, prompts=None, batch_size=64, return_scores=False):
    """prompts: optional list to use instead of df['prompt'] (used for RAG-augmented
    prompts later — same row order as df)."""
    df = df.reset_index(drop=True)
    prompt_list = df["prompt"].tolist() if prompts is None else list(prompts)

    premises, hypotheses = [], []
    for i, row in df.iterrows():
        for opt in options:
            premises.append(prompt_list[i])
            hypotheses.append(f"This example is {row[opt]}.")

    scores = batched_entailment_scores(premises, hypotheses, batch_size=batch_size)
    scores = np.array(scores).reshape(len(df), len(options))

    rankings = np.argsort(-scores, axis=1)
    preds = [" ".join(np.array(options)[r][:3]) for r in rankings]
    if return_scores:
        return preds, scores
    return preds

In [42]:
val_preds_nli = predict_top3_nli(val, batch_size=64)
zero_score = mapk(val['answer'], val_preds_nli)
print("Zero-Shot mAP@3 (validation):", round(zero_score, 4))

Zero-Shot mAP@3 (validation): 0.5604


In [43]:
comparison = pd.DataFrame({
    "Model": ["Random baseline", "TF-IDF", "Word2Vec", "MiniLM", "Zero-Shot (BART-MNLI)"],
    "mAP@3": [random_score, tfidf_score, w2v_score, transformer_score, zero_score],
}).sort_values("mAP@3", ascending=False).reset_index(drop=True)

comparison

,Model,mAP@3
0,Zero-Shot (BART-MNLI),0.560417
1,MiniLM,0.392500
2,Word2Vec,0.337500
3,Random baseline,0.320833
4,TF-IDF,0.282083


# Milestone 3: Context Augmentation with RAG Pipelines

## Why RAG? Limitations of general LLMs

Pretrained LLMs (and the zero-shot NLI model we just used) answer purely from
what was baked into their weights during training. This causes a few concrete
problems for our MCQ task:

- **No access to specific/current facts** — if a question hinges on a detail
  the model never saw enough of during pretraining, it has no way to look it
  up; it can only guess based on surface patterns in the prompt and options.
- **Hallucination** — when uncertain, the model can produce fluent but
  incorrect reasoning instead of admitting it doesn't know.
- **Fixed knowledge, fixed context** — the model can't consult anything
  outside its input window, so obscure or niche questions are especially hard.

**Retrieval-Augmented Generation (RAG)** addresses this by giving the model
extra, relevant text *at inference time* instead of relying only on what's in
its weights. The pipeline is:

1. **Retrieve** — turn the question into a search query, fetch relevant
   passages from an external knowledge source (here: Wikipedia) and embed
   them into a vector index.
2. **Augment** — pick the top-k most similar passages to the question and
   prepend them to the prompt as context.
3. **Generate/Predict** — feed `context + prompt + choices` into the model
   so it can ground its answer in retrieved facts rather than guessing blind.

The next few cells implement exactly these three steps.

## Turn a verbose MCQ prompt into a search query

In [44]:


# sanity check
for p in val['prompt'].head(3):
    print(repr(p))
    print("->", repr(extract_query(p)))
    print()

'what is a phageome'
-> 'what is a phageome'

'identify the correct statement what is the role of cycloidea genes in the evolution of bilateral symmetry carefully'
-> 'what is the role of cycloidea genes in the evolution of bilateral symmetry'

'determine the correct option what did fresnel predict and verify with regards to total internal reflections'
-> 'what did fresnel predict and verify with regards to total internal reflections'



## Build the vector database

In [45]:
!pip install -q wikipedia faiss-cpu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 63.8 MB/s eta 0:00:00


In [46]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import faiss

HEADERS = {"User-Agent": "DLGenAIProject/1.0 (23f3000717)"}

def fetch_passages(query, max_chars=6000, chunk_sentences=3):
    """Search Wikipedia for `query`, pull the plain-text article body (not just the
    short intro summary), and chunk it into small overlapping passages for retrieval.
    """
    try:
        search_url = "https://en.wikipedia.org/w/api.php"
        search_params = {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srlimit": 1,
            "format": "json",
        }
        r = requests.get(search_url, params=search_params, headers=HEADERS, timeout=10)
        r.raise_for_status()
        results = r.json().get("query", {}).get("search", [])
        if not results:
            return []
        title = results[0]["title"]

        # pull the full plain-text article body instead of just the lead summary —
        # the specific fact a question is testing is often further down the page
        extract_params = {
            "action": "query",
            "prop": "extracts",
            "explaintext": 1,
            "titles": title,
            "format": "json",
        }
        r = requests.get(search_url, params=extract_params, headers=HEADERS, timeout=10)
        r.raise_for_status()
        pages = r.json().get("query", {}).get("pages", {})
        text = next(iter(pages.values()), {}).get("extract", "")
        text = text[:max_chars]

        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.strip()) > 20]
        passages = [
            " ".join(sentences[i:i + chunk_sentences])
            for i in range(0, len(sentences), chunk_sentences)
        ]
        return [p for p in passages if p]

    except Exception as e:
        print(f"{query} -> {e}")
        return []

In [47]:
def build_vector_db(df, encoder, cache=None, max_workers=8):
    """Fetch Wikipedia passages for every unique query in `df`, embed them, and
    build a FAISS index. `cache` (dict query -> passages) is optional and lets you
    reuse fetched passages instead of hitting Wikipedia again.
    """
    cache = {} if cache is None else cache
    queries = [extract_query(p) for p in df["prompt"]]
    unique_queries = [q for q in dict.fromkeys(queries) if q not in cache]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_query = {executor.submit(fetch_passages, q): q for q in unique_queries}
        for future in tqdm(as_completed(future_to_query), total=len(future_to_query),
                            desc="Fetching Wikipedia context"):
            q = future_to_query[future]
            try:
                cache[q] = future.result()
            except Exception:
                cache[q] = []

    passage_texts = []
    query_to_passage_idx = {}
    for q in dict.fromkeys(queries):
        idxs = []
        for p in cache.get(q, []):
            idxs.append(len(passage_texts))
            passage_texts.append(p)
        query_to_passage_idx[q] = idxs

    if not passage_texts:
        print("WARNING: no passages retrieved for this batch — check that Internet "
              "access is enabled (Notebook Settings -> Internet -> On). RAG steps below "
              "will fall back to the non-RAG prediction automatically.")
        return None, [], {}, cache

    embeddings = encoder.encode(
        passage_texts,
        convert_to_numpy=True,
        batch_size=64,
        show_progress_bar=False,
        normalize_embeddings=True,  # so inner product == cosine similarity
    )

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    return index, passage_texts, query_to_passage_idx, cache

In [48]:
import os
import pickle
import faiss

VECTOR_DB_DIR = "/kaggle/working/vector_db"
os.makedirs(VECTOR_DB_DIR, exist_ok=True)

def build_or_load_vector_db(df, encoder, cache_name, wiki_cache=None, max_workers=8):
    """Load a pre-built vector database from disk if one exists for `cache_name`;
    otherwise build it from Wikipedia once and persist it for next time."""
    index_path = f"{VECTOR_DB_DIR}/{cache_name}.index"
    meta_path = f"{VECTOR_DB_DIR}/{cache_name}.pkl"

    if os.path.exists(index_path) and os.path.exists(meta_path):
        print(f"Loading pre-built vector database: {cache_name}")
        index = faiss.read_index(index_path)
        with open(meta_path, "rb") as f:
            passage_texts, query_to_passage_idx, wiki_cache = pickle.load(f)
        return index, passage_texts, query_to_passage_idx, wiki_cache

    print(f"No pre-built vector database found for '{cache_name}' — building it now...")
    index, passage_texts, query_to_passage_idx, wiki_cache = build_vector_db(
        df, encoder, cache=wiki_cache, max_workers=max_workers
    )

    if index is not None:
        faiss.write_index(index, index_path)
        with open(meta_path, "wb") as f:
            pickle.dump((passage_texts, query_to_passage_idx, wiki_cache), f)
        print(f"Saved vector database to {index_path} for future reuse.")

    return index, passage_texts, query_to_passage_idx, wiki_cache

In [49]:
wiki_cache = {}  # query -> list[passage]; reused for validation and, later, the test set

vector_db, passage_texts, query_to_passage_idx, wiki_cache = build_or_load_vector_db(
    val, minilm, cache_name="val", wiki_cache=wiki_cache
)

print("Vector database size (passages):", 0 if vector_db is None else vector_db.ntotal)

No pre-built vector database found for 'val' — building it now...


Fetching Wikipedia context:   0%|          | 0/220 [00:00<?, ?it/s]

what is the application of memristor -> 429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&prop=extracts&explaintext=1&titles=Memristor&format=json
what is the role of cycloidea genes in the evolution of bilateral symmetry -> 429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&prop=extracts&explaintext=1&titles=Symmetry+in+biology&format=json
what is the main focus of the environmental science center at qatar university -> 429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&prop=extracts&explaintext=1&titles=Qatar+University&format=json
which of the following is correct who was the first person to describe the pulmonary circulation framework -> 429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=which+of+the+following+is+correct+who+was+the+first+person+to+describe+the+pulmonary+circulation+framework&sr

## Retrieve context for a question

In [50]:
def retrieve_context_batch(prompts, encoder, index, passage_texts, k=5, batch_size=64):
    if index is None or not passage_texts:
        return ["" for _ in prompts]

    queries = [extract_query(p) for p in prompts]
    query_embs = encoder.encode(
        queries, convert_to_numpy=True, batch_size=batch_size,
        show_progress_bar=False, normalize_embeddings=True,
    )
    _, idxs = index.search(query_embs, min(k, index.ntotal))
    contexts = []
    for row_idxs in idxs:
        retrieved = [passage_texts[i] for i in row_idxs if i != -1]
        contexts.append(" ".join(retrieved))
    return contexts


sample_prompt = val.loc[0, "prompt"]
sample_context = retrieve_context_batch([sample_prompt], minilm, vector_db, passage_texts)[0]
print("PROMPT:", sample_prompt)
print()
print("RETRIEVED CONTEXT:", sample_context[:500], "...")

PROMPT: pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options

RETRIEVED CONTEXT: In the special case of isotropic media, the secondary wavefronts must be spherical, and Huygens's construction then implies that the rays are perpendicular to the wavefront; indeed, the law of ordinary refraction can be separately derived from that premise, as Ignace-Gaston Pardies did before Huygens. Although Newton rejected the wave theory, he noticed its potential to explain colors, including the colors of "thin plates" (e.g., "Newton's rings", and the colors of skylight reflected in soap bub ...


##  Feed `context + prompt + choices` into the model

In [51]:
def make_rag_prompts(df, encoder, index, passage_texts, k=5, max_context_chars=600):
    contexts = retrieve_context_batch(df["prompt"].tolist(), encoder, index, passage_texts, k=k)
    return [
        f"Context: {c[:max_context_chars]}\nQuestion: {p}" if c else p
        for c, p in zip(contexts, df["prompt"])
    ]

In [52]:
val_rag_prompts = make_rag_prompts(val, minilm, vector_db, passage_texts)

val_preds_rag = predict_top3_nli(val, prompts=val_rag_prompts, batch_size=64)
rag_score = mapk(val['answer'], val_preds_rag)
print("RAG + Zero-Shot mAP@3 (full validation set):", round(rag_score, 4))
print("Zero-Shot (no context) mAP@3, same set:", round(zero_score, 4))

RAG + Zero-Shot mAP@3 (full validation set): 0.5246
Zero-Shot (no context) mAP@3, same set: 0.5604


In [53]:
BEST_METHOD = "zero_shot"  # one of: "rag", "ensemble", "zero_shot" — set based on the comparison table above

test_vector_db, test_passage_texts, test_query_to_passage_idx, wiki_cache = build_or_load_vector_db(
    test, minilm, cache_name="test", wiki_cache=wiki_cache
)
print("Test vector database size (passages):", 0 if test_vector_db is None else test_vector_db.ntotal)

No pre-built vector database found for 'test' — building it now...


Fetching Wikipedia context:   0%|          | 0/96 [00:00<?, ?it/s]

what is the role of axioms in a formal theory -> 429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=what+is+the+role+of+axioms+in+a+formal+theory&srlimit=1&format=jsonwhat is the estimated redshift of ceers93316 a candidate highredshift galaxy observed by the james webb space telescope -> 429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=what+is+the+estimated+redshift+of+ceers93316+a+candidate+highredshift+galaxy+observed+by+the+james+webb+space+telescope&srlimit=1&format=json
what is the significance of the redshiftdistance relationship in determining the expansion history of the universe -> 429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=what+is+the+significance+of+the+redshiftdistance+relationship+in+determining+the+expansion+history+of+the+universe&srlimit=1&format=json
which of the following

## Updated model comparison

In [54]:
comparison = pd.DataFrame({
    "Model": [
        "Random baseline", "TF-IDF", "Word2Vec", "MiniLM",
        "Zero-Shot (BART-MNLI)", "Zero-Shot + RAG (Wikipedia)",
    ],
    "mAP@3": [random_score, tfidf_score, w2v_score, transformer_score, zero_score, rag_score],
}).sort_values("mAP@3", ascending=False).reset_index(drop=True)

comparison

,Model,mAP@3
0,Zero-Shot (BART-MNLI),0.560417
1,Zero-Shot + RAG (Wikipedia),0.524583
2,MiniLM,0.392500
3,Word2Vec,0.337500
4,Random baseline,0.320833
5,TF-IDF,0.282083


In [55]:
def zscore_rows(mat):
    mean = mat.mean(axis=1, keepdims=True)
    std = mat.std(axis=1, keepdims=True) + 1e-8
    return (mat - mean) / std

def cosine_sim_matrix(df, encoder, prompts=None, batch_size=64):
    prompt_list = df['prompt'].tolist() if prompts is None else list(prompts)
    prompt_emb = encoder.encode(prompt_list, convert_to_numpy=True, batch_size=batch_size, show_progress_bar=False)
    option_embs = {
        opt: encoder.encode(df[opt].tolist(), convert_to_numpy=True, batch_size=batch_size, show_progress_bar=False)
        for opt in options
    }
    return np.column_stack([cosine_similarity(prompt_emb, option_embs[opt]).diagonal() for opt in options])

_, nli_val_scores = predict_top3_nli(val, prompts=val_rag_prompts, batch_size=64, return_scores=True)
cos_val_scores = cosine_sim_matrix(val, minilm, prompts=val_rag_prompts)

ensemble_scores = zscore_rows(nli_val_scores) + zscore_rows(cos_val_scores)
ensemble_rankings = np.argsort(-ensemble_scores, axis=1)
ensemble_preds = [" ".join(np.array(options)[r][:3]) for r in ensemble_rankings]
ensemble_score = mapk(val['answer'], ensemble_preds)
print("Ensemble (NLI + MiniLM cosine, both with RAG context) mAP@3:", round(ensemble_score, 4))

Ensemble (NLI + MiniLM cosine, both with RAG context) mAP@3: 0.4725


In [56]:
nli_preds, nli_test_scores = predict_top3_nli(val, prompts=val_rag_prompts, batch_size=64, return_scores=True)
cos_val_scores = cosine_sim_matrix(val, minilm, prompts=val_rag_prompts)
combined = zscore_rows(nli_test_scores) + zscore_rows(cos_val_scores)
rankings = np.argsort(-combined, axis=1)
val_preds_ensemble = [" ".join(np.array(options)[r][:3]) for r in rankings]

ensemble_score = mapk(val['answer'], val_preds_ensemble)
print("Ensemble (NLI + MiniLM cosine, both with RAG context) mAP@3:", round(ensemble_score, 4))



Ensemble (NLI + MiniLM cosine, both with RAG context) mAP@3: 0.4725


In [57]:
comparison = pd.DataFrame({
    "Model": list(comparison["Model"]) + ["Ensemble (NLI + cosine, RAG context)"],
    "mAP@3": list(comparison["mAP@3"]) + [ensemble_score],
}).sort_values("mAP@3", ascending=False).reset_index(drop=True)

comparison

,Model,mAP@3
0,Zero-Shot (BART-MNLI),0.560417
1,Zero-Shot + RAG (Wikipedia),0.524583
2,"Ensemble (NLI + cosine, RAG context)",0.472500
3,MiniLM,0.392500
4,Word2Vec,0.337500
5,Random baseline,0.320833
6,TF-IDF,0.282083


# Milestone 4: Formulating the MCQ Task & LoRA Fine-Tuning

So far every method has been *training-free*: TF-IDF, Word2Vec, MiniLM embeddings,
zero-shot NLI, and RAG all use frozen models and never update a single weight on our
own data. That is a strong baseline, but the model has never actually *seen* what a
"correct answer" looks like for **these** questions.

In this milestone we change that. We:

1. **Reformulate the MCQ task** as a text-classification problem the model can be
   trained on — concatenating the question with each option to build `(prompt, label)`
   pairs.
2. **Fine-tune** a pretrained transformer on the training split — but instead of
   full fine-tuning (updating *all* ~110M weights), we use **LoRA** (Low-Rank
   Adaptation), which trains a tiny fraction of extra parameters and leaves the base
   model frozen.
3. Set up a proper **training loop** with the Hugging Face `Trainer`, manage **GPU
   memory / batch sizes**, and apply **efficiency strategies** (mixed precision,
   gradient accumulation, etc.).

We reuse everything already defined earlier: `train`, `test`, `tr`, `val`, `options`,
`SEED`, `DEVICE`, `apk`, `mapk`, and `log_run`.

In [58]:
!pip install -q "transformers>=4.40" "peft>=0.11" "accelerate>=0.30" datasets sentencepiece

## 1. Formulating the MCQ task: concatenating question + options

A pretrained encoder like BERT does not natively "understand" a 5-way multiple-choice
question. We have to turn each MCQ into something it *can* be trained on.

The standard trick is to **explode** each question into 5 independent
`(question + one option)` pairs and frame the problem as **binary classification**:
*"does this option correctly answer this question — yes or no?"*

So one MCQ row (with options A–E and gold answer, say, `B`) becomes 5 training rows:

| text                                | label |
|-------------------------------------|-------|
| `Question: ... [SEP] Option: <A>`   | 0     |
| `Question: ... [SEP] Option: <B>`   | **1** |
| `Question: ... [SEP] Option: <C>`   | 0     |
| `Question: ... [SEP] Option: <D>`   | 0     |
| `Question: ... [SEP] Option: <E>`   | 0     |

At inference we run all 5 pairs for a question, take the model's probability of the
`1` ("correct") class as a score per option, and rank them — exactly the same
"score-each-option-then-rank" pattern used by the NLI model in Milestone 2/3. This
keeps the whole pipeline (and the `mapk` scorer) compatible.

In [59]:
# We strip the boilerplate prefixes/suffixes ("Pick the best possible answer:", "carefully.")
# so the model focuses on the actual question. extract_query() from Milestone 3 already does
# this cleanly, so we reuse it.

def build_pairs(df, with_labels=True):
    """Explode an MCQ dataframe into (text, label) rows: one row per (question, option).

    Returns a dict of parallel lists so it maps straight into a datasets.Dataset.
    `example_id` / `option` let us regroup the 5 rows per question at inference time.
    """
    texts, labels, example_ids, opt_letters = [], [], [], []
    df = df.reset_index(drop=True)
    for i, row in df.iterrows():
        question = extract_query(row['prompt'])   # cleaned question stem
        for opt in options:
            texts.append(f"Question: {question}\nOption: {row[opt]}")
            example_ids.append(int(row['id']))
            opt_letters.append(opt)
            if with_labels:
                labels.append(1 if row['answer'] == opt else 0)

    out = {"text": texts, "example_id": example_ids, "option": opt_letters}
    if with_labels:
        out["label"] = labels
    return out


# Build the exploded train / validation sets from the SAME split used everywhere else
train_pairs = build_pairs(tr,  with_labels=True)
val_pairs   = build_pairs(val, with_labels=True)

print("Train pairs:", len(train_pairs['text']), "  (=", len(tr), "questions x 5 options)")
print("Val   pairs:", len(val_pairs['text']),   "  (=", len(val), "questions x 5 options)")
print("\nExample positive pair:\n", train_pairs['text'][train_pairs['label'].index(1)][:300], "...")
print("Label balance (train):", np.bincount(train_pairs['label']), "-> 1-in-5 positives, as expected")

Train pairs: 8000   (= 1600 questions x 5 options)
Val   pairs: 2000   (= 400 questions x 5 options)

Example positive pair:
 Question: what is modified newtonian dynamics mond
Option: mond is a hypothesis that proposes a modification of newtons law of universal gravitation to account for observed properties of galaxies it is an alternative to the hypothesis of dark matter in terms of explaining why galaxies do not appear  ...
Label balance (train): [6400 1600] -> 1-in-5 positives, as expected


In [60]:
from datasets import Dataset
from transformers import AutoTokenizer

# A small, fast encoder — good accuracy/speed trade-off for fine-tuning on a single GPU.
BASE_MODEL = "distilbert-base-uncased"
MAX_LENGTH = 256   # options can be long; 256 covers almost all pairs without wasting memory

ft_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_batch(batch):
    return ft_tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        # NOTE: no padding here — we pad *dynamically* per-batch via a data collator,
        # which is far more memory-efficient than padding everything to MAX_LENGTH.
    )

train_ds = Dataset.from_dict(train_pairs).map(tokenize_batch, batched=True, remove_columns=["text"])
val_ds   = Dataset.from_dict(val_pairs).map(tokenize_batch,   batched=True, remove_columns=["text"])

# Keep only what the model needs as tensors; example_id/option stay as plain columns for regrouping.
train_ds = train_ds.remove_columns(["example_id", "option"])
print(train_ds)
print("A tokenized example keys:", list(train_ds[0].keys()))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 8000
})
A tokenized example keys: ['label', 'input_ids', 'token_type_ids', 'attention_mask']


## 2. LoRA fine-tuning vs. full fine-tuning

**Full fine-tuning** updates *every* weight in the model. For DistilBERT that's ~67M
parameters; for a full BERT-base ~110M. This is:

- **Memory-hungry** — the optimizer (Adam) keeps 2 extra copies of *every* trainable
  parameter (momentum + variance), so trainable-param memory roughly triples. Add
  activations for backprop and even a "small" model can OOM a modest GPU.
- **Storage-hungry** — you save a full copy of the model per experiment.
- **Prone to catastrophic forgetting** on small datasets like ours (~1.6k questions).

**LoRA (Low-Rank Adaptation)** takes a different route. It **freezes the entire
pretrained model** and injects tiny trainable "adapter" matrices into the attention
layers. The key insight: the *update* to a big weight matrix `W (d×k)` during
fine-tuning is empirically low-rank, so we can approximate it as a product of two
skinny matrices `B (d×r)` and `A (r×k)` with rank `r ≪ d`:

$$W' = W + \Delta W = W + \frac{\alpha}{r}\, B A$$

Only `A` and `B` are trained. With `r = 8` on a 768-dim model, each adapted matrix adds
`768·8 + 8·768 ≈ 12k` params instead of `768·768 ≈ 590k` — a ~50× reduction.

**Advantages of LoRA over full fine-tuning:**

| Aspect | Full fine-tuning | LoRA |
|---|---|---|
| Trainable params | 100% (~67–110M) | typically **<1%** |
| Optimizer-state memory | 2× all params | 2× *only the adapters* |
| GPU memory | High (often OOM) | Fits on modest GPUs |
| Checkpoint size | Full model (100s of MB) | Adapters only (a few MB) |
| Catastrophic forgetting | More likely | Base weights untouched → less |
| Swapping tasks | Reload whole model | Swap tiny adapters |

The trade-off: LoRA has slightly less capacity than full fine-tuning, so on huge
datasets full FT can edge ahead — but for our data size, LoRA is the clear choice.

In [61]:
import torch
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

# Load the base classifier (2 labels: 0 = wrong option, 1 = correct option)
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
)

# --- LoRA configuration ---
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,   # sequence classification head stays trainable
    r=32,                         # rank of the low-rank update — the key capacity knob
    lora_alpha=64,                # scaling (alpha/r = 2.0 effective scale)
    lora_dropout=0.1,
    target_modules=["q_lin", "k_lin", "v_lin", "out_lin"],
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.to(DEVICE)

# This one line makes the whole LoRA pitch concrete:
model.print_trainable_parameters()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,771,778 || all params: 68,726,788 || trainable%: 2.5780


## 3. The training loop, GPU memory & efficiency strategies

We use Hugging Face's `Trainer`, which wraps the standard PyTorch training loop
(forward → loss → `backward()` → `optimizer.step()` → `zero_grad()`, batched over
epochs) so we don't hand-write it. All the tuning happens through **`TrainingArguments`**.

**Managing GPU memory and batch size.** The single biggest memory lever is
`per_device_train_batch_size`. If you OOM, halve it. To keep the *effective* batch size
large (which stabilises training) without the memory cost, we use
**gradient accumulation**: process several small micro-batches and only step the
optimizer once their gradients are summed.

> effective batch size = `per_device_train_batch_size` × `gradient_accumulation_steps`

**Efficiency strategies baked into the arguments below:**

- **`fp16` mixed precision** (on GPU) — ~2× faster, ~half the activation memory.
- **Dynamic padding** via a data collator — pad each batch to its own longest example,
  not to `MAX_LENGTH`, avoiding wasted compute on padding tokens.
- **Gradient accumulation** — big effective batch, small memory footprint.
- **Evaluate/save per epoch + `load_best_model_at_end`** — keep the checkpoint that
  actually generalises best, not just the last one.
- LoRA itself is the biggest efficiency win — almost no optimizer state to store.

In [62]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from transformers import EarlyStoppingCallback

# Dynamic padding: pads each batch to its own max length (memory-efficient)
data_collator = DataCollatorWithPadding(tokenizer=ft_tokenizer)

# Compute the REAL objective (mAP@3) during eval so we keep the best-ranking checkpoint,
# not just the one best at rejecting wrong options.
# Assumes val_ds rows are in question order (5 consecutive rows/question) and eval isn't shuffled.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    assert len(logits) == len(val) * len(options), \
        f"eval size {len(logits)} != {len(val)}*{len(options)} — order/alignment broken"
    correct_probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    scores = correct_probs.reshape(len(val), len(options))
    rankings = np.argsort(-scores, axis=1)
    preds = [" ".join(np.array(options)[r][:3]) for r in rankings]
    return {"map3": float(mapk(val['answer'].reset_index(drop=True), preds))}

# DeBERTa-v3-base is ~3x DistilBERT: if you OOM, set batch=8 and grad_accum=4 (effective batch stays 32)
training_args = TrainingArguments(
    output_dir="/kaggle/working/lora_mcq",
    num_train_epochs=8,
    per_device_train_batch_size=16,      # lower to 8/4 if you hit CUDA OOM
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,       # effective batch size = 32
    learning_rate=1e-4,                  # LoRA likes a higher LR than full FT (~2e-5)
    warmup_ratio=0.1,                   # deprecation warning is cosmetic — safe to keep
    weight_decay=0.01,
    fp16=USE_CUDA,
    bf16=False,                       # mixed precision when a GPU is present
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="map3",        # select checkpoint on real mAP@3, not pair-accuracy
    greater_is_better=True,
    logging_steps=25,
    report_to="none",                    # we log the final mAP@3 to W&B ourselves via log_run
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=ft_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# Free any leftover cached memory before the big allocation of training
if USE_CUDA:
    torch.cuda.empty_cache()
    print("GPU mem before train:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


GPU mem before train: 1.2 GB


In [63]:
# --- Fine-tune! ---
trainer.train()

if USE_CUDA:
    print("Peak GPU mem during train:", round(torch.cuda.max_memory_allocated()/1e9, 2), "GB")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Map3
1,1.976406,1.005684,0.624167
2,1.887124,0.904685,0.713750
3,1.407103,0.649746,0.860833
4,1.145561,0.517091,0.930417
5,0.861566,0.376241,0.943333
6,0.809765,0.315199,0.953750
7,0.805492,0.273762,0.954167
8,0.688501,0.280891,0.956667


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

Peak GPU mem during train: 3.04 GB


## 4. Evaluating the fine-tuned model with mAP@3

Now we score each option with the model's probability of the **"correct" (label 1)**
class, regroup the 5 exploded rows back into their original questions, rank the options,
and feed the top-3 into the same `mapk` scorer used for every other method — so the
number is directly comparable to the earlier milestones.

In [64]:
import torch.nn.functional as F

@torch.no_grad()
def predict_top3_finetuned(df, model, tokenizer, batch_size=64, max_length=MAX_LENGTH):
    """Score every (question, option) pair with the fine-tuned model and rank options."""
    model.eval()
    pairs = build_pairs(df, with_labels=False)
    texts = pairs["text"]
    correct_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, truncation=True, max_length=max_length,
                        padding=True, return_tensors="pt").to(DEVICE)
        logits = model(**enc).logits
        probs = F.softmax(logits, dim=-1)[:, 1]     # P(label == 1 == "correct")
        correct_probs.extend(probs.cpu().numpy().tolist())
    scores = np.array(correct_probs).reshape(len(df), len(options))
    rankings = np.argsort(-scores, axis=1)
    preds = [" ".join(np.array(options)[r][:3]) for r in rankings]
    return preds, scores

val_preds_ft, _ = predict_top3_finetuned(val, model, ft_tokenizer, batch_size=64)
finetuned_score = mapk(val['answer'], val_preds_ft)
print("LoRA fine-tuned mAP@3 (validation):", round(finetuned_score, 4))

# log to Weights & Biases, consistent with earlier milestones
try:
    log_run("LoRA fine-tuned (DeBERTa-v3)", finetuned_score,
            extra_config={"base_model": BASE_MODEL, "lora_r": lora_config.r,
                          "epochs": training_args.num_train_epochs})
except Exception as e:
    print("W&B logging skipped:", e)

LoRA fine-tuned mAP@3 (validation): 0.9567
W&B logging skipped: No API key configured. Use `wandb login` to log in.


In [65]:
# Add the fine-tuned model to the running comparison table
comparison = pd.DataFrame({
    "Model": list(comparison["Model"]) + ["LoRA fine-tuned (DeBERTa-v3)"],
    "mAP@3": list(comparison["mAP@3"]) + [finetuned_score],
}).sort_values("mAP@3", ascending=False).reset_index(drop=True)

comparison

,Model,mAP@3
0,LoRA fine-tuned (DeBERTa-v3),0.956667
1,Zero-Shot (BART-MNLI),0.560417
2,Zero-Shot + RAG (Wikipedia),0.524583
3,"Ensemble (NLI + cosine, RAG context)",0.472500
4,MiniLM,0.392500
5,Word2Vec,0.337500
6,Random baseline,0.320833
7,TF-IDF,0.282083


In [66]:
# Save ONLY the LoRA adapters (a few MB) — not the whole base model.
# This is one of LoRA's practical wins: tiny, portable checkpoints.
ADAPTER_DIR = "/kaggle/working/lora_mcq_adapter"
model.save_pretrained(ADAPTER_DIR)
ft_tokenizer.save_pretrained(ADAPTER_DIR)

import os
size_mb = sum(os.path.getsize(os.path.join(ADAPTER_DIR, f))
              for f in os.listdir(ADAPTER_DIR)) / 1e6
print(f"Saved LoRA adapter to {ADAPTER_DIR}  (~{size_mb:.1f} MB total)")
print("To reuse: load the base model, then PeftModel.from_pretrained(base, ADAPTER_DIR)")

Saved LoRA adapter to /kaggle/working/lora_mcq_adapter  (~7.8 MB total)
To reuse: load the base model, then PeftModel.from_pretrained(base, ADAPTER_DIR)


In [67]:
# Final model: retrain on ALL of train (tr + val) before predicting test.
# The tr-only `model` above is for measuring val mAP@3; for the actual submission
# we want every labeled question, so we refit from scratch on the full set.
full_pairs = build_pairs(train, with_labels=True)
full_ds = Dataset.from_dict(full_pairs).map(tokenize_batch, batched=True, remove_columns=["text"])
full_ds = full_ds.remove_columns(["example_id", "option"])

base_full = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)
model_full = get_peft_model(base_full, lora_config).to(DEVICE)

args_full = TrainingArguments(
    output_dir="/kaggle/working/lora_mcq_full",
    num_train_epochs=training_args.num_train_epochs,   # same epochs as the validated run
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=training_args.learning_rate,
    warmup_ratio=0.1, weight_decay=0.01,
    fp16=USE_CUDA, eval_strategy="no", save_strategy="no",
    logging_steps=25, report_to="none", seed=SEED,
)
Trainer(model=model_full, args=args_full, train_dataset=full_ds,
        processing_class=ft_tokenizer, data_collator=data_collator).train()
print("Full-data model trained.")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were

Step,Training Loss
25,2.737838
50,2.138849
75,1.986722
100,2.053213
125,2.004796
150,1.959745
175,2.039064
200,1.952965
225,1.974714
250,1.923642


Full-data model trained.


# sub

#sub

In [68]:
BEST_METHOD = "finetuned"  # one of: "rag", "ensemble", "zero_shot", "finetuned"

if BEST_METHOD in ("rag", "ensemble"):
    # only these methods need the Wikipedia vector DB
    test_vector_db, test_passage_texts, test_query_to_passage_idx, wiki_cache = build_vector_db(
        test, minilm, cache=wiki_cache
    )
    print("Test vector database size (passages):",
          0 if test_vector_db is None else test_vector_db.ntotal)
else:
    # zero_shot and finetuned don't retrieve context
    test_vector_db, test_passage_texts, test_query_to_passage_idx = None, [], {}
    print(f"BEST_METHOD='{BEST_METHOD}' needs no vector DB — skipping Wikipedia build.")

BEST_METHOD='finetuned' needs no vector DB — skipping Wikipedia build.


In [69]:
if BEST_METHOD == "zero_shot":
    test_predictions = predict_top3_nli(test, batch_size=64)

elif BEST_METHOD == "finetuned":
    test_predictions, _ = predict_top3_finetuned(test, model_full, ft_tokenizer, batch_size=64)

elif BEST_METHOD == "rag":
    test_rag_prompts = make_rag_prompts(test, minilm, test_vector_db, test_passage_texts)
    test_predictions = predict_top3_nli(test, prompts=test_rag_prompts, batch_size=64)

else:  # ensemble
    test_rag_prompts = make_rag_prompts(test, minilm, test_vector_db, test_passage_texts)
    nli_preds, nli_test_scores = predict_top3_nli(test, prompts=test_rag_prompts, batch_size=64, return_scores=True)
    cos_test_scores = cosine_sim_matrix(test, minilm, prompts=test_rag_prompts)
    combined = zscore_rows(nli_test_scores) + zscore_rows(cos_test_scores)
    rankings = np.argsort(-combined, axis=1)
    test_predictions = [" ".join(np.array(options)[r][:3]) for r in rankings]

submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": test_predictions,
})

submission.to_csv("/kaggle/working/submission.csv", index=False)
submission.to_csv("/kaggle/working/sub.csv", index=False)

print("Saved /kaggle/working/submission.csv and /kaggle/working/sub.csv")
print("Prediction letter counts (top choice only):")
print(submission["Prediction"].str.split().str[0].value_counts())
submission.head()

Saved /kaggle/working/submission.csv and /kaggle/working/sub.csv
Prediction letter counts (top choice only):
Prediction
B    114
C    111
D     98
A     90
E     87
Name: count, dtype: int64


,ID,Prediction
0,1,A B E
1,2,B E A
2,3,B E D
3,4,E C A
4,5,D C E


#new